In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HOQ_J.xpt"
csv_file = DATA_DIR / "HOQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
# For HOQ_J, NO numeric correction should ultimately
# be necessary.
#
source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 HOQ_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HOD050",
    "HOQ065"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "HOQ_J column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 9254:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 3:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,254 rows x 3 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# CDC HOQ_J variables:
#
# SEQN
#     Respondent sequence number
#
# HOD050
#     Number of rooms in home
#
# HOQ065
#     Home owned / bought / rented / other
#
# Attached HOQ_J contains NO genuine continuous
# decimal variables.
#
# All 3 variables are proven whole-number fields.
#
# IMPORTANT:
#
# This rule applies ONLY to HOQ_J.
#
# Genuine Dietary, Laboratory, Examination,
# sample-weight, or other continuous decimal fields
# in other NHANES files must remain untouched.
#

DECIMAL_COLUMNS = []


INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Whole-number/code variables:",
    INTEGER_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known SAS XPORT / pandas decoding artifact:
#
#     5.397605346934028e-79
#
# Some NHANES files contain this value where CDC
# intended numeric zero.
#
# HOQ_J was inspected directly.
#
# Expected for this attachment:
#
#     SEQN   = 0 tiny values
#     HOD050 = 0 tiny values
#     HOQ065 = 0 tiny values
#
# Therefore NO tiny-value replacement should be done.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 3 HOQ_J variables "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-VALUE CHECK ---")

print(
    tiny_counts.to_string()
)

print(
    "Total tiny XPORT artifacts:",
    tiny_total
)


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in HOQ_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: HOQ_J contains no tiny XPORT artifacts."
)


# ============================================================
# 6. CHECK ORIGINAL ORDINARY ZERO PATTERN
# ============================================================

#
# Attached HOQ_J contains no legitimate numeric zero
# codes either.
#
# This is useful because it proves that no zero repair
# or preservation issue exists in this specific file.
#

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL NUMERIC ZERO CHECK ---")

print(
    true_zero_counts.to_string()
)

print(
    "Ordinary numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zero values "
        "found in HOQ_J."
    )


print(
    "PASS: Original HOQ_J contains no numeric zeros."
)


# ============================================================
# 7. EXPECTED CORRECTED SOURCE
# ============================================================

#
# Unlike HUQ_J, there is NO validated numeric repair
# required for HOQ_J.
#
# Therefore expected_df must remain numerically
# identical to source_df.
#

expected_df = source_df.copy()


# ============================================================
# 8. CHECK FOR GENUINE FRACTIONAL VARIABLES
# ============================================================

#
# Run BEFORE converting anything to Int64.
#
# np.round() is used ONLY as a comparison test.
# It does NOT modify the data.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables unexpectedly "
        "found in HOQ_J.\n"
        "They will NOT be rounded or converted to Int64:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 9. VALIDATE SEQN BEFORE CONVERSION
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- ORIGINAL SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 10. CONVERT PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# All 3 fields have now been proven whole-number.
#
# NO:
#
#     .round()
#
# is used during conversion.
#
# Int64 changes CSV representation:
#
#     93703.0 -> 93703
#     6.0     -> 6
#
# but does NOT change their numeric values.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 3 HOQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. CDC FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC CHECK - HOD050
# ============================================================

#
# Number of rooms in home.
#
# CDC notes that reported values >=13 were grouped
# and released as category 13.
#

check_frequency(
    "HOD050",
    {
        1: 78,
        2: 198,
        3: 713,
        4: 1693,
        5: 1848,
        6: 1562,
        7: 1115,
        8: 670,
        9: 415,
        10: 276,
        11: 79,
        12: 51,
        13: 47,
        777: 8,
        999: 27
    },
    474
)


# ============================================================
# 13. CDC CHECK - HOQ065
# ============================================================

#
# Home owned, being bought, rented,
# or other arrangement.
#

check_frequency(
    "HOQ065",
    {
        1: 4871,
        2: 3690,
        3: 202,
        7: 11,
        9: 6
    },
    474
)


# ============================================================
# 14. ALLOWED CDC CODE CHECK
# ============================================================

ALLOWED_VALUES = {

    "HOD050": {
        1, 2, 3, 4, 5, 6, 7,
        8, 9, 10, 11, 12, 13,
        777, 999
    },

    "HOQ065": {
        1, 2, 3, 7, 9
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )

    unexpected = (
        observed
        -
        allowed
    )

    if unexpected:

        raise ValueError(
            f"{col} contains unexpected CDC codes: "
            f"{sorted(unexpected)}"
        )

    print(
        f"PASS: {col} contains only "
        "CDC-documented values."
    )


# ============================================================
# 15. MISSING-PATTERN CONSISTENCY CHECK
# ============================================================

#
# In the attached HOQ_J:
#
# HOD050 missing = 474
# HOQ065 missing = 474
#
# The missing positions are exactly the same.
#
# This is a useful additional integrity check.
#

hod050_missing = (
    df["HOD050"]
    .isna()
    .to_numpy()
)


hoq065_missing = (
    df["HOQ065"]
    .isna()
    .to_numpy()
)


missing_pattern_mismatches = int(
    np.sum(
        hod050_missing
        !=
        hoq065_missing
    )
)


print("\n--- HOUSING VARIABLE MISSING PATTERN ---")

print(
    "Missing-position mismatches:",
    missing_pattern_mismatches
)


if missing_pattern_mismatches != 0:

    raise ValueError(
        "HOD050 and HOQ065 missing positions "
        "unexpectedly differ."
    )


if int(df["HOD050"].isna().sum()) != 474:

    raise ValueError(
        "Unexpected HOD050 missing count."
    )


if int(df["HOQ065"].isna().sum()) != 474:

    raise ValueError(
        "Unexpected HOQ065 missing count."
    )


print(
    "PASS: HOD050 and HOQ065 share exactly "
    "the same 474 missing records."
)


# ============================================================
# 16. FINAL ZERO PATTERN
# ============================================================

#
# No zeros existed in the source.
# No zero restoration was required.
#
# Therefore final zero pattern must remain empty.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    zero_count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if zero_count > 0:

        final_zero_counts[col] = zero_count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected zero values were introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zero values introduced."
)


# ============================================================
# 17. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# For HOQ_J:
#
# PERMITTED NUMERIC MODIFICATIONS = NONE
#
# Therefore every current numeric value must exactly
# match source_df.
#
# Int64 conversion affects display/storage type only,
# not numeric value.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    source_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        source_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 3 variables exactly match "
    "their original XPT numeric values."
)


# ============================================================
# 18. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    original_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            original_missing
            !=
            current_missing
        )
    )


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value locations changed "
        "before CSV export."
    )


print(
    "PASS: Missing-value positions unchanged."
)


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# Fidelity-preserving export.
#
# IMPORTANT:
#
# NO:
#
#     .round()
#
# NO:
#
#     float_format=
#
# NO:
#
#     blanket decimal formatting
#
# Although HOQ_J has no genuine decimal variables,
# retaining this export policy is important for your
# full NHANES workflow.
#
# Dietary/Lab/Exam/etc. decimal variables in other
# files should appear from pandas without forced
# reformatting.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 20. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 21. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

#
# HOQ_J requires ZERO numeric corrections.
#
# Therefore exported values should round-trip
# exactly to original XPT-derived numeric values.
#

print(
    "\n--- EXACT XPT -> CSV NUMERIC FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 3 CSV variables round-trip "
    "to exact original XPT numeric values."
)


# ============================================================
# 22. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 23. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Reading as strings lets us validate the literal
# CSV representation.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 24. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT values found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in CSV."
)


# ============================================================
# 25. RAW CSV .0 CHECK
# ============================================================

#
# HOQ_J has NO genuine decimal columns.
#
# Therefore it is safe to check ALL 3 variables for
# unnecessary trailing ".0".
#
# Examples:
#
#     93703.0 -> 93703
#     6.0     -> 6
#     1.0     -> 1
#
# IMPORTANT:
#
# Do NOT apply this check indiscriminately to a file
# containing genuine decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' "
    "in any HOQ_J field."
)


# ============================================================
# 26. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zeros found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 27. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 28. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93703-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 29. FINAL FREQUENCY RECHECK FROM CSV
# ============================================================

#
# Confirm the exported CSV retains the exact CDC
# distribution.
#

final_hod050 = pd.to_numeric(
    raw_csv_df["HOD050"]
    .where(
        raw_csv_df["HOD050"] != "",
        np.nan
    ),
    errors="coerce"
)


final_hoq065 = pd.to_numeric(
    raw_csv_df["HOQ065"]
    .where(
        raw_csv_df["HOQ065"] != "",
        np.nan
    ),
    errors="coerce"
)


EXPECTED_HOD050 = {
    1: 78,
    2: 198,
    3: 713,
    4: 1693,
    5: 1848,
    6: 1562,
    7: 1115,
    8: 670,
    9: 415,
    10: 276,
    11: 79,
    12: 51,
    13: 47,
    777: 8,
    999: 27
}


EXPECTED_HOQ065 = {
    1: 4871,
    2: 3690,
    3: 202,
    7: 11,
    9: 6
}


actual_hod050 = (
    final_hod050
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


actual_hoq065 = (
    final_hoq065
    .dropna()
    .astype(int)
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual_hod050 != EXPECTED_HOD050:

    raise ValueError(
        "Final CSV HOD050 frequencies differ."
    )


if actual_hoq065 != EXPECTED_HOQ065:

    raise ValueError(
        "Final CSV HOQ065 frequencies differ."
    )


if int(final_hod050.isna().sum()) != 474:

    raise ValueError(
        "Final CSV HOD050 missing count differs."
    )


if int(final_hoq065.isna().sum()) != 474:

    raise ValueError(
        "Final CSV HOQ065 missing count differs."
    )


print(
    "PASS: Final CSV frequencies exactly "
    "match CDC HOQ_J."
)


# ============================================================
# 30. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "HOQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - "
    "Housing Characteristics"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "HOD050 missing:",
    int(
        df["HOD050"]
        .isna()
        .sum()
    )
)

print(
    "HOQ065 missing:",
    int(
        df["HOQ065"]
        .isna()
        .sum()
    )
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2017-2018 HOQ_J XPT -> CSV
Rows: 9,254
Columns: 3
PASS: Structure = 9,254 rows x 3 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Whole-number/code variables: ['SEQN', 'HOD050', 'HOQ065']

--- ORIGINAL XPT TINY-VALUE CHECK ---
SEQN      0
HOD050    0
HOQ065    0
Total tiny XPORT artifacts: 0
PASS: HOQ_J contains no tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
SEQN      0
HOD050    0
HOQ065    0
Ordinary numeric zeros: 0
PASS: Original HOQ_J contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
HOD050: 0 fractional observations
HOQ065: 0 fractional observations
PASS: No genuine fractional variables.

--- ORIGINAL SEQN ---
Minimum: 93703.0
Maximum: 102956.0
Missing: 0
Duplicates: 0
PASS: SEQN 93703-102956 validated.

PASS: All 3 HOQ_J variables converted to nullable Int64 without rounding.

--- HOD050 ---
Observed: {np.int64(1): 78, np.int64(2): 198, np.int64(3): 713, np.int64(4): 1693, np.int64(5): 1848, np.int64